# 23. 최종 지표 (v2.1): 원래 질문 + 오타 질문, 검증 + 시험

11/03 아침 최종 설정과 오늘 개선(20 오타 대응, 22 DPO v2)을 **같은 질문·같은 문단·같은 심판**으로 한 번에 비교합니다.

| 축 | 비교 |
|---|---|
| 생성 | Qwen3-8B (DPO 없음) / + DPO v1 (17) / + DPO v2 (22) |
| 거절 | A: MiniLM < 4.92 / B2: 미달이면 질문 고쳐 재검색, 고친 질문은 6.0 이상이어야 답 (20) |
| 질문 | 원래 / 흔들린 (오타 1~2개 + 문법 흔들기, 20과 같은 규칙·시드) |
| 평가셋 | 검증 (답 있음 200 + 답 없음 100) = 고르기 / 시험 650 = 보고 |

- 검색 결과는 20 (`typo_gate/retrieval.json`), 심판 캐시는 14~22와 공유
- 결과: `final_eval/results_summary.md`, 추천 `final_eval/final_choice.json`

## 1. 설치 + 경로

In [ ]:
!pip -q install -U peft faiss-cpu sentence-transformers
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (14·15에서 확인한 조합)

In [ ]:
QUICK = False   # True: 질문 수십 개로 점검 (결과는 final_eval/quick) / False: 본 실행

import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'          # 08 시험 답변
R2_DIR     = f'{SAVE_DIR}/rerank2'      # 11 후보·재정렬 점수
JUDGE_DIR  = f'{SAVE_DIR}/judge_eval'   # 14 검증 답변 + 심판 캐시
TG_DIR     = f'{SAVE_DIR}/typo_gate'
V1_DIR     = f'{SAVE_DIR}/dpo'
V2_DIR     = f'{SAVE_DIR}/dpo_v2'
OUT_DIR    = f'{SAVE_DIR}/final_eval' + ('/quick' if QUICK else '')
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(REPORT_DIR, exist_ok=True)
print('QUICK =', QUICK, '→', OUT_DIR)

## 2. 공통 함수

In [ ]:
import re, json, gzip, time, random, string, hashlib, gc
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch, transformers, peft

SEED      = 42
THRESH    = 4.921875
ENSEMBLE  = ['bge-reranker-large', 'bge-reranker-v2-m3']
BASE_RR   = 'ms-marco-MiniLM-L-6-v2'
BASE_GEN  = 'Qwen/Qwen2.5-7B-Instruct'
GENS      = ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B']
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
JUDGE2    = 'microsoft/phi-4'
GEN_BS    = 16
N_TEST_A, N_TEST_N, N_TEST_O, N_VAL = (20, 5, 5, 12) if QUICK else (None, None, None, None)
DEMO = ['why does my face turn red and itchy after drinking alcohol',
        'what are the symptoms of low vitamin d',
        'can ibuprofen cause stomach bleeding',
        'how long does the flu last',
        'what is a normal resting heart rate for adults']

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')
print(f'torch {torch.__version__} / transformers {transformers.__version__} / peft {peft.__version__}')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f'GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())
docs_text = lambda pids: '\n'.join(f'[{i}] ({sites[p]}) {texts[p]}' for i, p in enumerate(pids, 1))
def rag_messages(q, pids):
    return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{docs_text(pids)}\n\nQuestion: {q}'}]
print(f'문단 {len(texts):,}')
RETRY_TH = 6.0
# GPU 메모리가 작으면(L4 등) 생성 배치를 줄여 OOM 방지
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
if GPU_GB < 40:
    GEN_BS = 4
    print(f'GPU {GPU_GB:.0f}GB → 생성 배치 {GEN_BS}. 심판(14B)은 A100(40GB 이상) 필요')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_gen(name):
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
    return tok, llm

@torch.inference_mode()
def generate(tok, llm, msgs_list, bs=GEN_BS, max_new=256):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                   for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        outs += [re.sub(r'<think>.*?</think>', '', o, flags=re.S).strip() for o in dec]
    return outs

def run_system(tok, llm, key, docs_key):
    # key = '검색|모델' 시스템 답변을 시험·검증 모두 생성 (거절 규칙 08과 같음)
    for rows in (test_rows, val_rows):
        todo = [r for r in rows if not r['refuse'] and key not in r['ans']]
        t = time.time()
        outs = generate(tok, llm, [rag_messages(r['query'], r['docs'][docs_key]) for r in todo])
        for r, a in zip(todo, outs):
            r['ans'][key] = a
        for r in rows:
            if r['refuse']:
                r['ans'][key] = REFUSAL
        if todo:
            print(f'  {key}: {len(todo)}개 생성 {time.time() - t:.0f}초')
    save_answers()

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')
SYS_EVID = ('You check whether the given documents contain the information needed to answer a health question. '
            'Reply with one digit only.\n'
            '1 = at least one document directly answers the question\n'
            '0 = the documents do not answer the question (related topic only, or missing information)')
def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]
def faith_msgs(dt, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]
def evid_msgs(q, dt):
    return [{'role': 'system', 'content': SYS_EVID},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nQuestion: {q}\n\nDo the documents answer it (0 or 1)?'}]

class Judge:
    def __init__(self, name, cache_path, tag=''):
        self.tok = AutoTokenizer.from_pretrained(name, padding_side='left')
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
        self.digits = [self.tok.encode(d, add_special_tokens=False)[-1] for d in '012']
        self.path, self.tag = cache_path, tag
        self.cache = {j['key']: j for j in load_jsonl(cache_path)} if os.path.exists(cache_path) else {}
    def key(self, kind, q, a):
        return hashlib.md5(f'{self.tag}{kind}|{q}|{a}'.encode()).hexdigest()
    @torch.inference_mode()
    def probs(self, msgs_list, n, bs=8):
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
        order = np.argsort([len(p) for p in prompts]); out = [None] * len(prompts)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = self.tok([prompts[i] for i in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(self.llm.device)
            logits = self.llm(**enc).logits[:, -1, self.digits[:n]].float()
            for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
                out[i] = p
        return out
    def score(self, kind, items):
        # 'c' (q, refs, ans) → 0~2 / 'f' (q, docs_text, ans) → 0~1 / 'e' (q, docs_text) → 0~1
        keyed = [(self.key(kind, it[0], it[-1]), it) for it in items]
        need = list({k: it for k, it in keyed if k not in self.cache}.items())
        t = time.time()
        for s in range(0, len(need), 2000):
            part = need[s:s + 2000]
            if kind == 'c':
                vals = [float(p[1] + 2 * p[2]) for p in self.probs([correct_msgs(*it) for _, it in part], 3)]
            elif kind == 'f':
                vals = [float(p[1]) for p in self.probs([faith_msgs(it[1], it[2]) for _, it in part], 2)]
            else:
                vals = [float(p[1]) for p in self.probs([evid_msgs(it[0], it[1]) for _, it in part], 2)]
            for (k, _), v in zip(part, vals):
                self.cache[k] = {'key': k, 'v': v}
            save_jsonl(list(self.cache.values()), self.path)
        if need:
            print(f'  심판{self.tag} {kind}: {len(need)}개 새로 채점 ({time.time() - t:.0f}초)')
        return [self.cache[k]['v'] for k, _ in keyed]
    def close(self):
        del self.llm; free_gpu()

def evaluate(J, rows, systems, faith=True):
    table = {}
    for key in systems:
        dk = key.split('|')[0]
        A = [r for r in rows if r['group'] == 'A']
        ansA = [r for r in A if not is_refusal(r['ans'][key])]
        cs = dict(zip([id(r) for r in ansA], J.score('c', [(r['query'], r['answers'], r['ans'][key]) for r in ansA])))
        c_all = [cs.get(id(r), 0.0) for r in A]
        g = lambda k: [r for r in rows if r['group'] == k]
        row = {'정답률 (심판 2점)': np.mean([c >= 1.5 for c in c_all]),
               '정답+부분': np.mean([c >= 0.5 for c in c_all]),
               '평균 점수 (0~2)': np.mean(c_all),
               'F1 (참고)': np.mean([max(f1(r['ans'][key], x) for x in r['answers']) for r in A]),
               '잘못 거절': np.mean([is_refusal(r['ans'][key]) for r in A])}
        if faith:
            answered = [r for r in rows if not is_refusal(r['ans'][key])]
            fs = J.score('f', [(r['query'], docs_text(r['docs'][dk]), r['ans'][key]) for r in answered])
            row['근거 충실도'] = np.mean([f >= 0.5 for f in fs]) if fs else np.nan
        if g('N'):
            row['답 없음 거절률 (MS MARCO 기준)'] = np.mean([is_refusal(r['ans'][key]) for r in g('N')])
        if g('O'):
            row['분야 밖 거절률'] = np.mean([is_refusal(r['ans'][key]) for r in g('O')])
        table[key] = row
    return pd.DataFrame(table).T.round(3)

JUDGE_PATH = f'{JUDGE_DIR}/judgments.jsonl'

## 3. 흔들린 질문 (20과 같음)

In [ ]:
KEYS = 'qwertyuiopasdfghjklzxcvbnm'
def typo_word(w, rng):
    i = rng.randrange(1, len(w) - 1)
    op = rng.choice(['swap', 'drop', 'dup', 'sub'])
    if op == 'swap':
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    if op == 'drop':
        return w[:i] + w[i + 1:]
    if op == 'dup':
        return w[:i] + w[i] + w[i:]
    return w[:i] + rng.choice(KEYS) + w[i + 1:]

FILLER = {'the', 'a', 'an', 'is', 'are', 'do', 'does', 'of'}
def make_noisy(q, seed):
    rng = random.Random(f'noisy|{seed}')
    words = q.split()
    idx = [i for i, w in enumerate(words) if len(w) >= 4 and w.isalpha()]
    for i in rng.sample(idx, min(len(idx), 1 if len(words) < 6 else 2)):
        words[i] = typo_word(words[i], rng)
    out = [w for w in words if not (w.lower() in FILLER and rng.random() < 0.5)] or words
    s = ' '.join(out)
    if rng.random() < 0.3:
        s = re.sub(r'\b(symptoms|causes|signs|effects|side effects|treatments)\b', lambda m: m.group(1)[:-1], s)
    return s.lower() if rng.random() < 0.5 else s

for q in ['what are the symptoms of influenza', 'is sugar bad for your teeth', 'what is a heart lung bypass machine']:
    print(q, '→', make_noisy(q, q))

## 4. 질문 + 검색 (20 결과 재사용)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
valgen   = load_jsonl(f'{JUDGE_DIR}/valgen_set.jsonl')
vN = [r for r in load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl') if r['group'] == 'N']

rows = []
for q in eval_set:
    g = 'A' if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids') else ('N' if q['domain'] == 'health' and not q.get('answerable', True) else ('O' if q['domain'] != 'health' else None))
    if g:
        rows.append({'set': 'test', 'qid': str(q['qid']), 'query': q['query'], 'answers': q['answers'], 'gold': q.get('gold_pids') or [], 'group': g})
for q in val_set:
    if q.get('gold_pids'):
        rows.append({'set': 'val', 'qid': str(q['qid']), 'query': q['query'], 'answers': q['answers'], 'gold': q['gold_pids'], 'group': 'A'})
for r in vN:
    rows.append({'set': 'val', 'qid': str(r['qid']), 'query': r['query'], 'answers': [], 'gold': [], 'group': 'N'})
if QUICK:
    pick_ = lambda s, g, n: [r for r in rows if r['set'] == s and r['group'] == g][:n]
    rows = pick_('test', 'A', 24) + pick_('test', 'N', 6) + pick_('test', 'O', 4) + pick_('val', 'A', 16) + pick_('val', 'N', 6)
for r in rows:
    r['noisy'] = make_noisy(r['query'], r['qid'])
print(Counter((r['set'], r['group']) for r in rows))

index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
emb = SentenceTransformer('BAAI/bge-base-en-v1.5', device='cuda').half()
def ce_load(name, max_len):
    try:
        m = CrossEncoder(name, max_length=max_len, device='cuda', activation_fn=torch.nn.Identity())
    except TypeError:
        m = CrossEncoder(name, max_length=max_len, device='cuda', default_activation_function=torch.nn.Identity())
    m.model.half()
    return m
RR = {'mini': ce_load('cross-encoder/ms-marco-MiniLM-L-6-v2', 256),
      'large': ce_load('BAAI/bge-reranker-large', 512), 'm3': ce_load('BAAI/bge-reranker-v2-m3', 512)}
zs = lambda m: (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def search(queries, k=100):
    qe = emb.encode(['Represent this sentence for searching relevant passages: ' + q for q in queries], batch_size=128,
                    normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    return index.search(qe, k)[1]

def pick(order_scores, ids, n):
    docs, seen = [], set()
    for j in np.argsort(-order_scores):
        p = int(ids[j]); key = norm_text(texts[p])
        if key in seen:
            continue
        seen.add(key); docs.append(p)
        if len(docs) == n:
            break
    return docs

def full_retrieve(queries):
    ids = search(queries)
    pairs = [(q, texts[p]) for q, c in zip(queries, ids.tolist()) for p in c]
    sc = {n: m.predict(pairs, batch_size=256 if n == 'mini' else 64, show_progress_bar=len(queries) > 200).reshape(len(queries), 100)
          for n, m in RR.items()}
    ens = (zs(sc['large']) + zs(sc['m3'])) / 2
    return [{'ids': ids[i].tolist(), 'mini': float(sc['mini'][i].max()), 'm3': float(sc['m3'][i].max()),
             'E10': pick(ens[i], ids[i], 10)} for i in range(len(queries))]

RET_PATH = f'{TG_DIR}/retrieval.json'
if os.path.exists(RET_PATH):
    saved = json.load(open(RET_PATH))
    for r in rows:
        r['ret'] = saved[r['set'] + '|' + r['qid']]
    print('저장된 검색 결과 불러옴')
else:
    t = time.time()
    for v in ('query', 'noisy'):
        res = full_retrieve([r[v] for r in rows])
        for r, x in zip(rows, res):
            r.setdefault('ret', {})[v] = x
    json.dump({r['set'] + '|' + r['qid']: r['ret'] for r in rows}, open(RET_PATH, 'w'))
    print(f'검색 끝 {(time.time() - t) / 60:.1f}분')

def recall(rows_, v, key):
    A = [r for r in rows_ if r['group'] == 'A' and r['gold']]
    if key == 'ids':
        return np.mean([bool(set(r['gold']) & set(r['ret'][v]['ids'])) for r in A])
    return np.mean([bool(set(r['gold']) & set(r['ret'][v][key])) for r in A])
for s in ('val', 'test'):
    R = [r for r in rows if r['set'] == s]
    print(f"[{s}] 정답 문단 포함률  후보100: 원래 {recall(R, 'query', 'ids'):.3f} / 흔들림 {recall(R, 'noisy', 'ids'):.3f}"
          f"  ·  상위10: 원래 {recall(R, 'query', 'E10'):.3f} / 흔들림 {recall(R, 'noisy', 'E10'):.3f}")
    A = [r for r in R if r['group'] == 'A']
    print(f"      MiniLM 최고점 중앙값: 원래 {np.median([r['ret']['query']['mini'] for r in A]):.2f} / 흔들림 {np.median([r['ret']['noisy']['mini'] for r in A]):.2f}"
          f"  ·  4.92 통과율: 원래 {np.mean([r['ret']['query']['mini'] >= THRESH for r in A]):.3f} / 흔들림 {np.mean([r['ret']['noisy']['mini'] >= THRESH for r in A]):.3f}")

## 5. 질문 고쳐 재검색 (점수 미달만, 20의 고친 질문 재사용)

In [ ]:
from peft import PeftModel
SYS_FIX = ('Rewrite the user question as a clear English health search query. Fix spelling mistakes and replace casual '
           'phrases with standard medical terms when obvious (e.g. "sympton" -> "symptoms", "shot after rusty nail" -> '
           '"tetanus shot after stepping on a rusty nail"). Do not answer it. Output only the rewritten question.')
GEN = 'Qwen/Qwen3-8B'
tok, llm = load_gen(GEN)
llm = PeftModel.from_pretrained(llm, f'{V1_DIR}/adapter', adapter_name='v1')
HAS_V2 = os.path.exists(f'{V2_DIR}/adapter/adapter_config.json')
if HAS_V2:
    llm.load_adapter(f'{V2_DIR}/adapter', adapter_name='v2')
llm.eval()
print('DPO v2 있음:', HAS_V2)

@torch.inference_mode()
def rewrite_many(qs, bs=32):
    out = []
    with llm.disable_adapter():
        for i in range(0, len(qs), bs):
            prompts = [tok.apply_chat_template([{'role': 'system', 'content': SYS_FIX}, {'role': 'user', 'content': q}],
                                               tokenize=False, add_generation_prompt=True, enable_thinking=False) for q in qs[i:i + bs]]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=60, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.pad_token_id or tok.eos_token_id)
            dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
            out += [re.sub(r'<think>.*?</think>', '', d, flags=re.S).strip().split('\n')[0].strip('" ') for d in dec]
    return out

prev = {}
if os.path.exists(f'{TG_DIR}/rewrites.jsonl'):
    for d in load_jsonl(f'{TG_DIR}/rewrites.jsonl'):
        prev[d['qid']] = d
for v in ('query', 'noisy'):
    need = []
    for r in rows:
        if r['ret'][v]['mini'] >= THRESH:
            continue
        p = prev.get(r['qid']) if r['set'] == 'test' else None
        if p and p.get(f'fix_{v}'):
            r[f'fix_{v}'] = p[f'fix_{v}']
        else:
            need.append(r)
    for r, f in zip(need, rewrite_many([r[v] for r in need])):
        r[f'fix_{v}'] = f
    todo = [r for r in rows if f'fix_{v}' in r]
    res = full_retrieve([r[f'fix_{v}'] for r in todo]) if todo else []
    for r, x in zip(todo, res):
        r[f'fixret_{v}'] = x
    print(v, '고쳐 재검색', len(todo))

def decide(r, v, gate):
    x = r['ret'][v]
    if x['mini'] >= THRESH:
        return r[v], x['E10']
    if gate == 'B2' and f'fixret_{v}' in r and r[f'fixret_{v}']['mini'] >= RETRY_TH:
        return r[f'fix_{v}'], r[f'fixret_{v}']['E10']
    return None

## 6. 답 생성 (어댑터 × 거절 × 질문, 같은 입력은 한 번만)

In [ ]:
ADAPTERS = ['none', 'v1'] + (['v2'] if HAS_V2 else [])
GATES = ['A', 'B2']
ANS_PATH = f'{OUT_DIR}/answers.jsonl'
cache = {d['k']: d['a'] for d in load_jsonl(ANS_PATH)} if os.path.exists(ANS_PATH) else {}
ck = lambda ad, q, docs: hashlib.md5(f'{ad}|{q}|{docs}'.encode()).hexdigest()
for ad in ADAPTERS:
    jobs = {}
    for r in rows:
        for v in ('query', 'noisy'):
            for g in GATES:
                d = decide(r, v, g)
                if d and ck(ad, *d) not in cache:
                    jobs[ck(ad, *d)] = d
    if not jobs:
        continue
    ks = list(jobs); t = time.time()
    msgs = [rag_messages(jobs[k][0], jobs[k][1]) for k in ks]
    if ad == 'none':
        with llm.disable_adapter():
            outs = generate(tok, llm, msgs)
    else:
        llm.set_adapter(ad); outs = generate(tok, llm, msgs)
    cache.update(dict(zip(ks, outs)))
    save_jsonl([{'k': k, 'a': a} for k, a in cache.items()], ANS_PATH)
    print(f'{ad}: {len(ks)}개 생성 {(time.time() - t) / 60:.1f}분')
del llm; free_gpu()

## 7. 심판 + 표

In [ ]:
assert GPU_GB >= 40, '답은 answers.jsonl에 저장됨. 심판 Qwen2.5-14B(bf16 약 30GB)는 A100에서만 실행: 런타임 유형을 A100으로 바꿔 모두 실행하면 생성은 건너뛰고 채점만 합니다.'
J = Judge(JUDGE_LLM, JUDGE_PATH)
def ans(r, v, ad, g):
    d = decide(r, v, g)
    return REFUSAL if d is None else cache[ck(ad, *d)]
items = {}
for r in rows:
    if r['group'] != 'A':
        continue
    for v in ('query', 'noisy'):
        for ad in ADAPTERS:
            for g in GATES:
                a = ans(r, v, ad, g)
                if not is_refusal(a):
                    items[(r['query'], a)] = (r['query'], r['answers'], a)
J.score('c', list(items.values()))
N = [r for r in rows if r['group'] == 'N']
evid = dict(zip([(r['set'], r['qid']) for r in N], J.score('e', [(r['query'], docs_text(r['ret']['query']['E10'])) for r in N])))
fitems = {}
for r in rows:
    for v in ('query', 'noisy'):
        for ad in ADAPTERS:
            for g in GATES:
                d = decide(r, v, g); a = ans(r, v, ad, g)
                if d and not is_refusal(a):
                    fitems[(r['query'], a)] = (r['query'], docs_text(d[1]), a)
J.score('f', list(fitems.values()))
cv = lambda r, a: 0.0 if is_refusal(a) else J.cache[J.key('c', r['query'], a)]['v']
fv = lambda r, a: J.cache[J.key('f', r['query'], a)]['v']

NAME = {'none': 'Qwen3-8B', 'v1': '+ DPO v1', 'v2': '+ DPO v2'}
def table(s):
    R = [r for r in rows if r['set'] == s]
    A = [r for r in R if r['group'] == 'A']; Nn = [r for r in R if r['group'] == 'N' and evid[(s, r['qid'])] < 0.2]; O = [r for r in R if r['group'] == 'O']
    out = {}
    for ad in ADAPTERS:
        for g in GATES:
            row = {}
            for v, lab in (('query', '원래'), ('noisy', '오타')):
                aa = {r['qid']: ans(r, v, ad, g) for r in R}
                row[f'정답률 ({lab})'] = np.mean([cv(r, aa[r['qid']]) >= 1.5 for r in A])
                row[f'잘못 거절 ({lab})'] = np.mean([is_refusal(aa[r['qid']]) for r in A])
                answered = [r for r in R if not is_refusal(aa[r['qid']])]
                row[f'근거 충실도 ({lab})'] = np.mean([fv(r, aa[r['qid']]) >= 0.5 for r in answered]) if answered else np.nan
                row[f'진짜 답 없음 거절 ({lab})'] = np.mean([is_refusal(aa[r['qid']]) for r in Nn]) if Nn else np.nan
                if O:
                    row[f'분야 밖 거절 ({lab})'] = np.mean([is_refusal(aa[r['qid']]) for r in O])
            out[f'{NAME[ad]} · 거절 {g}'] = row
    return pd.DataFrame(out).T.round(3), len(A), len(Nn)
tv, nA_v, nN_v = table('val'); print(f'[검증: 답 있음 {nA_v}, 진짜 답 없음 {nN_v}]'); display(tv)
tt, nA_t, nN_t = table('test'); print(f'[시험: 답 있음 {nA_t}, 진짜 답 없음 {nN_t}]'); display(tt)

# 고르기 (검증): 원래·오타 정답률 평균 최고, 단 진짜 답 없음 거절(원래)이 아침 설정(DPO v1 · A)보다 0.05 넘게 낮지 않을 것
base_key = '+ DPO v1 · 거절 A'
score = (tv['정답률 (원래)'] + tv['정답률 (오타)']) / 2
ok = tv[tv['진짜 답 없음 거절 (원래)'] >= tv.loc[base_key, '진짜 답 없음 거절 (원래)'] - 0.05]
CHOICE = score[ok.index].idxmax()
print('→ 검증으로 고른 설정:', CHOICE)

# 시험: 고른 설정 vs 아침 설정 정답률 차이 (원래·오타) 부트스트랩
A_t = [r for r in rows if r['set'] == 'test' and r['group'] == 'A']
inv = {f'{NAME[ad]} · 거절 {g}': (ad, g) for ad in ADAPTERS for g in GATES}
def boot(key, base, v):
    d = np.array([(cv(r, ans(r, v, *inv[key])) >= 1.5) - (cv(r, ans(r, v, *inv[base])) >= 1.5) for r in A_t], dtype=float)
    bs = np.random.default_rng(SEED).choice(d, (2000, len(d))).mean(1)
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
ci = pd.DataFrame({f'{CHOICE} − 아침 설정 ({lab})': dict(zip(['차이', '95% CI 하한', '95% CI 상한'], boot(CHOICE, base_key, v)))
                   for v, lab in (('query', '원래'), ('noisy', '오타'))}).T.round(3)
display(ci)
J.close(); del J

## 8. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
ad, g = inv[CHOICE]
json.dump({'choice': CHOICE, 'adapter': {'none': None, 'v1': 'rag_msmarco/health/dpo/adapter', 'v2': 'rag_msmarco/health/dpo_v2/adapter'}[ad],
           'retry': g == 'B2', 'retry_threshold': RETRY_TH if g == 'B2' else None}, open(f'{OUT_DIR}/final_choice.json', 'w'), ensure_ascii=False, indent=1)
summary = '\n'.join([
    f'# 23 최종 지표 (v2.1){" [QUICK 점검]" if QUICK else ""} ({stamp} KST)', '',
    '- 같은 질문·문단·심판(Qwen2.5-14B). 오타 질문은 20과 같은 규칙. 거절 A = MiniLM < 4.92, B2 = 미달이면 질문 고쳐 재검색(고친 질문 ≥ 6.0)',
    f'- **검증으로 고른 설정: {CHOICE}** (원래·오타 정답률 평균 최고, 진짜 답 없음 거절은 아침 설정 − 0.05 이내)', '',
    f'## 검증 (답 있음 {nA_v}, 진짜 답 없음 {nN_v})', '', tv.to_markdown(), '',
    f'## 시험 (답 있음 {nA_t}, 진짜 답 없음 {nN_t}, 분야 밖 50)', '', tt.to_markdown(), '',
    '## 시험 정답률 차이 (95% 부트스트랩)', '', ci.to_markdown(),
])
paths = [f'{OUT_DIR}/results_summary.md'] + ([] if QUICK else [f'{REPORT_DIR}/23_final_eval_summary.md'])
for p in paths:
    with open(p, 'w', encoding='utf-8') as f:
        f.write(summary)
print(summary)

## 9. 드라이브 저장 마무리 + 런타임 종료

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if not QUICK:
    runtime.unassign()